<a href="https://colab.research.google.com/github/RitAgrawal18/Trade-Tariff-Analysis/blob/main/notebook2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install linearmodels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.7/115.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 4.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings

# Suppress pandas future warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# 1. Data Cleaning
try:
    df = pd.read_csv('India_Trade_Tariff_Data_Cleaned.csv')
except FileNotFoundError:
    print("Error: 'India_Trade_Tariff_Data_Cleaned.csv' not found. Please ensure the file is in the working directory.")
    exit()

# Check for missing values
print("Missing Values:\n", df.isnull().sum())

# Ensure correct data types
df['Year'] = df['Year'].astype(int)
df['Trade_Partner'] = df['Trade_Partner'].astype('category')
numeric_cols = ['Export_Value_USD_Billion', 'Import_Value_USD_Billion', 'Trade_Deficit_USD_Billion',
                'Partner_Tariff_On_India_Percent', 'India_Tariff_On_Partner_Percent',
                'Pharma_Exports_USD_Billion', 'Auto_Exports_USD_Billion', 'Textile_Exports_USD_Billion',
                'Electronics_Exports_USD_Billion', 'Petroleum_Exports_USD_Billion',
                'Auto_Tariff_Percent', 'Textile_Tariff_Percent', 'Pharma_Tariff_Percent',
                'Log_Export_Value', 'Log_Auto_Exports', 'Log_Textile_Exports', 'Log_Pharma_Exports',
                'Z_Partner_Tariff']
df[numeric_cols] = df[numeric_cols].astype(float)

# Verify no NaN/infinite values in log columns
for col in ['Log_Export_Value', 'Log_Auto_Exports', 'Log_Textile_Exports', 'Log_Pharma_Exports']:
    if df[col].isna().sum() > 0 or np.isinf(df[col]).sum() > 0:
        print(f"Warning: {col} contains NaN or infinite values, which may affect regression results.")

# Add US_Treatment for interaction term
df['US_Treatment'] = (df['Trade_Partner'] == 'USA').astype(int)
df['Tariff_US_Interaction'] = df['Z_Partner_Tariff'] * df['US_Treatment']

# Set panel structure
df_panel = df.set_index(['Trade_Partner', 'Year'])

# 2. OLS Regression
ols_formula = ('Log_Export_Value ~ Z_Partner_Tariff + Auto_Tariff_Percent + Textile_Tariff_Percent + '
               'Tariff_US_Interaction')
ols_model = smf.ols(ols_formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['Trade_Partner']})
print("\nOLS Regression Results (Log Export Value):\n")
print(ols_model.summary())

# Plot OLS coefficients
coef = ols_model.params
errors = ols_model.bse
fig, ax = plt.subplots(figsize=(10, 6))
coef[1:].plot(kind='bar', yerr=errors[1:], ax=ax, color='lightblue', capsize=5)
ax.set_title('OLS Regression Coefficients (Log Export Value)')
ax.set_ylabel('Coefficient (Elasticity)')
ax.set_xlabel('Variables')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('ols_coefficients.png')
plt.close()

# 3. Panel Regression
panel_formula = ('Log_Export_Value ~ Z_Partner_Tariff + Auto_Tariff_Percent + Textile_Tariff_Percent + '
                 'Tariff_US_Interaction + EntityEffects')
panel_model = PanelOLS.from_formula(panel_formula, data=df_panel, drop_absorbed=True).fit(
    cov_type='clustered', cluster_entity=True)
print("\nPanel Regression Results (Fixed Effects, Log Export Value):\n")
print(panel_model.summary)

# 4. Sectoral Regressions
# Auto Exports
auto_model = smf.ols('Log_Auto_Exports ~ Z_Partner_Tariff + Auto_Tariff_Percent + Tariff_US_Interaction',
                     data=df).fit(cov_type='cluster', cov_kwds={'groups': df['Trade_Partner']})
print("\nAuto Exports OLS Results:\n")
print(auto_model.summary())

# Textile Exports
textile_model = smf.ols('Log_Textile_Exports ~ Z_Partner_Tariff + Textile_Tariff_Percent + Tariff_US_Interaction',
                        data=df).fit(cov_type='cluster', cov_kwds={'groups': df['Trade_Partner']})
print("\nTextile Exports OLS Results:\n")
print(textile_model.summary())

# Pharma Exports
pharma_model = smf.ols('Log_Pharma_Exports ~ Z_Partner_Tariff + Pharma_Tariff_Percent + Tariff_US_Interaction',
                       data=df).fit(cov_type='cluster', cov_kwds={'groups': df['Trade_Partner']})
print("\nPharma Exports OLS Results:\n")
print(pharma_model.summary())

# 5. Difference-in-Differences: 2022-2024
df['Export_Growth'] = df.groupby('Trade_Partner')['Export_Value_USD_Billion'].pct_change() * 100
df['Post_2024'] = (df['Year'] == 2024).astype(int)
df['Interaction'] = df['Post_2024'] * df['US_Treatment']

# Parallel trends test (pre-2024)
pre_2024 = df[df['Year'] < 2024].copy()
pre_2024['Year_Centered'] = pre_2024['Year'] - 2022
parallel_model = smf.ols('Export_Growth ~ Year_Centered * US_Treatment',
                         data=pre_2024.dropna(subset=['Export_Growth'])).fit()
print("\nParallel Trends Test (Pre-2024):\n")
print(parallel_model.summary())

# DiD regression
# DiD regression
did_formula = 'Export_Growth ~ Post_2024 + US_Treatment + Interaction + Z_Partner_Tariff + Auto_Tariff_Percent + Textile_Tariff_Percent'
# Subset the data for DiD regression and drop rows with missing 'Export_Growth'
did_data = df[df['Year'] >= 2022].dropna(subset=['Export_Growth'])
# Use the 'Trade_Partner' values from the subsetted data for clustering
did_model = smf.ols(did_formula, data=did_data).fit(
    cov_type='cluster', cov_kwds={'groups': did_data['Trade_Partner']})
print("\nDifference-in-Differences Results:\n")
print(did_model.summary())

# Plot export growth trends
plt.figure(figsize=(12, 6))
sns.lineplot(data=df, x='Year', y='Export_Growth', hue='Trade_Partner', marker='o', palette='tab10')
plt.axvline(x=2024, color='red', linestyle='--', label='Hypothetical U.S. Tariff Hike (2024)')
plt.title('Export Growth by Trade Partner (2022-2024)')
plt.ylabel('Export Growth (%)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('export_growth_trends.png')
plt.close()

# Plot parallel trends
pre_2024_pivot = pre_2024.pivot_table(values='Export_Growth', index='Year', columns='Trade_Partner')
plt.figure(figsize=(10, 6))
for partner in pre_2024_pivot.columns:
    plt.plot(pre_2024_pivot.index, pre_2024_pivot[partner], marker='o', label=partner)
plt.title('Pre-2024 Export Growth Trends (Parallel Trends Test)')
plt.ylabel('Export Growth (%)')
plt.xlabel('Year')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('parallel_trends.png')
plt.close()

# Save cleaned dataset
df.to_csv('India_Trade_Tariff_Data_Cleaned_Output.csv', index=False)
print("\nCleaned dataset saved as 'India_Trade_Tariff_Data_Cleaned_Output.csv'")

Missing Values:
 Year                               0
Trade_Partner                      0
Export_Value_USD_Billion           0
Import_Value_USD_Billion           0
Trade_Deficit_USD_Billion          0
Partner_Tariff_On_India_Percent    0
India_Tariff_On_Partner_Percent    0
Pharma_Exports_USD_Billion         0
Auto_Exports_USD_Billion           0
Textile_Exports_USD_Billion        0
Electronics_Exports_USD_Billion    0
Petroleum_Exports_USD_Billion      0
Auto_Tariff_Percent                0
Textile_Tariff_Percent             0
Pharma_Tariff_Percent              0
Log_Export_Value                   0
Log_Auto_Exports                   0
Log_Textile_Exports                0
Log_Pharma_Exports                 0
Z_Partner_Tariff                   0
dtype: int64

OLS Regression Results (Log Export Value):

                            OLS Regression Results                            
Dep. Variable:       Log_Export_Value   R-squared:                       0.797
Model:                     

/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 4, but rank is 2
  warnings.warn('covariance of constraints does not have full '
<ipython-input-2-30c090edb618>:69: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

Auto_Tariff_Percent, Textile_Tariff_Percent

  panel_model = PanelOLS.from_formula(panel_formula, data=df_panel, drop_absorbed=True).fit(
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)



Panel Regression Results (Fixed Effects, Log Export Value):

                          PanelOLS Estimation Summary                           
Dep. Variable:       Log_Export_Value   R-squared:                     -2.22e-16
Estimator:                   PanelOLS   R-squared (Between):             -7.1298
No. Observations:                  15   R-squared (Within):               0.0000
Date:                Wed, Apr 30 2025   R-squared (Overall):             -7.0982
Time:                        09:52:03   Log-likelihood                    18.288
Cov. Estimator:             Clustered                                           
                                        F-statistic:                  -1.629e-15
Entities:                           5   P-value                           1.0000
Avg Obs:                       3.0000   Distribution:                     F(1,9)
Min Obs:                       3.0000                                           
Max Obs:                       3.0000   F-stati

/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.11/dist-packages/statsmodels/regression/linear_model.py:1966: RuntimeWarning: divide by zero encountered in scalar divide
  return np.sqrt(eigvals[0]/eigvals[-1])
/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
  warnings.warn('covariance of constraints does not have full '
/usr/local/lib/python3.11/dist-packages/statsmodels/stats/stattools.py:74: ValueWarning: 

                            OLS Regression Results                            
Dep. Variable:     Log_Pharma_Exports   R-squared:                       0.718
Model:                            OLS   Adj. R-squared:                  0.671
Method:                 Least Squares   F-statistic:                     16.09
Date:                Wed, 30 Apr 2025   Prob (F-statistic):             0.0122
Time:                        09:52:03   Log-Likelihood:                -11.917
No. Observations:                  15   AIC:                             29.83
Df Residuals:                      12   BIC:                             31.96
Df Model:                           2                                         
Covariance Type:              cluster                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept                 0.90

In [3]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# OLS Regression: Simplified to reduce multicollinearity
ols_formula = 'Log_Export_Value ~ Auto_Tariff_Percent + Textile_Tariff_Percent + Tariff_US_Interaction'
ols_model = smf.ols(ols_formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['Trade_Partner']})
print("\nOLS Regression Results (Log Export Value):\n")
print(ols_model.summary())

# Check VIF for multicollinearity
X = df[['Auto_Tariff_Percent', 'Textile_Tariff_Percent', 'Tariff_US_Interaction']].copy()
X['Intercept'] = 1  # Add intercept for VIF
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\nVariance Inflation Factors:\n", vif_data)


OLS Regression Results (Log Export Value):

                            OLS Regression Results                            
Dep. Variable:       Log_Export_Value   R-squared:                       0.797
Model:                            OLS   Adj. R-squared:                  0.742
Method:                 Least Squares   F-statistic:                -1.419e+13
Date:                Wed, 30 Apr 2025   Prob (F-statistic):               1.00
Time:                        09:52:05   Log-Likelihood:                -10.395
No. Observations:                  15   AIC:                             28.79
Df Residuals:                      11   BIC:                             31.62
Df Model:                           3                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------

/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)
/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 3, but rank is 2
  warnings.warn('covariance of constraints does not have full '


In [4]:
# OLS Regression: Simplified to reduce multicollinearity
ols_formula = 'Log_Export_Value ~ Auto_Tariff_Percent + Tariff_US_Interaction'
ols_model = smf.ols(ols_formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['Trade_Partner']})
print("\nOLS Regression Results (Log Export Value):\n")
print(ols_model.summary())

# Plot OLS coefficients
coef = ols_model.params
errors = ols_model.bse
fig, ax = plt.subplots(figsize=(10, 6))
coef[1:].plot(kind='bar', yerr=errors[1:], ax=ax, color='lightblue', capsize=5)
ax.set_title('OLS Regression Coefficients (Log Export Value)')
ax.set_ylabel('Coefficient (Elasticity)')
ax.set_xlabel('Variables')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('ols_coefficients.png')
plt.close()

# VIF Analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = df[['Auto_Tariff_Percent', 'Tariff_US_Interaction']].copy()
X['Intercept'] = 1
vif_data = pd.DataFrame()
# OLS Regression: Simplified to reduce multicollinearity
ols_formula = 'Log_Export_Value ~ Auto_Tariff_Percent + Tariff_US_Interaction'
ols_model = smf.ols(ols_formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['Trade_Partner']})
print("\nOLS Regression Results (Log Export Value):\n")
print(ols_model.summary())

# Plot OLS coefficients
coef = ols_model.params
errors = ols_model.bse
fig, ax = plt.subplots(figsize=(10, 6))
coef[1:].plot(kind='bar', yerr=errors[1:], ax=ax, color='lightblue', capsize=5)
ax.set_title('OLS Regression Coefficients (Log Export Value)')
ax.set_ylabel('Coefficient (Elasticity)')
ax.set_xlabel('Variables')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('ols_coefficients.png')
plt.close()

# VIF Analysis
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = df[['Auto_Tariff_Percent', 'Tariff_US_Interaction']].copy()
X['Intercept'] = 1
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\nVariance Inflation Factors:\n", vif_data)


OLS Regression Results (Log Export Value):

                            OLS Regression Results                            
Dep. Variable:       Log_Export_Value   R-squared:                       0.415
Model:                            OLS   Adj. R-squared:                  0.317
Method:                 Least Squares   F-statistic:                     4.960
Date:                Wed, 30 Apr 2025   Prob (F-statistic):             0.0826
Time:                        09:52:05   Log-Likelihood:                -18.349
No. Observations:                  15   AIC:                             42.70
Df Residuals:                      12   BIC:                             44.82
Df Model:                           2                                         
Covariance Type:              cluster                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------

/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)



OLS Regression Results (Log Export Value):

                            OLS Regression Results                            
Dep. Variable:       Log_Export_Value   R-squared:                       0.415
Model:                            OLS   Adj. R-squared:                  0.317
Method:                 Least Squares   F-statistic:                     4.960
Date:                Wed, 30 Apr 2025   Prob (F-statistic):             0.0826
Time:                        09:52:05   Log-Likelihood:                -18.349
No. Observations:                  15   AIC:                             42.70
Df Residuals:                      12   BIC:                             44.82
Df Model:                           2                                         
Covariance Type:              cluster                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------

/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:430: UserWarning: `kurtosistest` p-value may be inaccurate with fewer than 20 observations; only n=15 observations were given.
  return hypotest_fun_in(*args, **kwds)



Variance Inflation Factors:
                 Variable        VIF
0    Auto_Tariff_Percent   2.506024
1  Tariff_US_Interaction   2.506024
2              Intercept  31.746988


In [5]:
import pandas as pd
import statsmodels.formula.api as smf
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Load data
df = pd.read_csv('India_Trade_Tariff_Data_Cleaned.csv')

# Simulate 2025 and 2026 data (5 rows each)
df_2025 = df[df['Year'] == 2024].copy()
df_2025['Year'] = 2025
df_2026 = df[df['Year'] == 2024].copy()
df_2026['Year'] = 2026
df = pd.concat([df, df_2025, df_2026], ignore_index=True)  # Now 25 rows

# Verify DataFrame size
if len(df) != 25:
    print(f"Error: Expected 25 rows, got {len(df)} rows.")
    exit()

# Add simulated control variables for 25 rows
df['India_GDP_Growth_Percent'] = [7.0]*5 + [7.2]*5 + [6.5]*5 + [6.2]*5 + [6.0]*5  # 2022–2026
df['INR_USD_Exchange_Rate'] = [78.6]*5 + [82.3]*5 + [83.5]*5 + [85.0]*5 + [86.0]*5  # 2022–2026
df['Z_GDP_Growth'] = (df['India_GDP_Growth_Percent'] - df['India_GDP_Growth_Percent'].mean()) / df['India_GDP_Growth_Percent'].std()
df['Z_Exchange_Rate'] = (df['INR_USD_Exchange_Rate'] - df['INR_USD_Exchange_Rate'].mean()) / df['INR_USD_Exchange_Rate'].std()

# Simulate 2025 data (e.g., U.S. tariff hike to 20%)
df.loc[df['Year'] == 2025, 'Partner_Tariff_On_India_Percent'] = [20.0 if tp == 'USA' else 5.0 for tp in df.loc[df['Year'] == 2025, 'Trade_Partner']]
df.loc[df['Year'] == 2025, 'Export_Value_USD_Billion'] *= [0.85 if tp == 'USA' else 1.0 for tp in df.loc[df['Year'] == 2025, 'Trade_Partner']]
# Simulate 2026 data (e.g., tariffs revert)
df.loc[df['Year'] == 2026, 'Partner_Tariff_On_India_Percent'] = [3.3 if tp == 'USA' else 5.0 for tp in df.loc[df['Year'] == 2026, 'Trade_Partner']]

# Add Tariff_US_Interaction
df['Tariff_US_Interaction'] = df['Partner_Tariff_On_India_Percent'] * (df['Trade_Partner'] == 'USA').astype(int)

# OLS Regression with controls
ols_formula = 'Log_Export_Value ~ Auto_Tariff_Percent + Tariff_US_Interaction + Z_GDP_Growth + Z_Exchange_Rate'
ols_model = smf.ols(ols_formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['Trade_Partner']})
print("\nOLS Regression Results (Log Export Value):\n")
print(ols_model.summary())

# VIF Analysis
X = df[['Auto_Tariff_Percent', 'Tariff_US_Interaction', 'Z_GDP_Growth', 'Z_Exchange_Rate']].copy()
X['Intercept'] = 1
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\nVariance Inflation Factors:\n", vif_data)


OLS Regression Results (Log Export Value):

                            OLS Regression Results                            
Dep. Variable:       Log_Export_Value   R-squared:                       0.341
Model:                            OLS   Adj. R-squared:                  0.209
Method:                 Least Squares   F-statistic:                     24.17
Date:                Wed, 30 Apr 2025   Prob (F-statistic):            0.00461
Time:                        09:52:06   Log-Likelihood:                -31.849
No. Observations:                  25   AIC:                             73.70
Df Residuals:                      20   BIC:                             79.79
Df Model:                           4                                         
Covariance Type:              cluster                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------

In [6]:
# OLS Regression: Remove Textile_Tariff_Percent to reduce multicollinearity
ols_formula = 'Log_Export_Value ~ Auto_Tariff_Percent + Z_GDP_Growth + Z_Exchange_Rate'
ols_model = smf.ols(ols_formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['Trade_Partner']})
print("\nOLS Regression Results (Log Export Value):\n")
print(ols_model.summary())

# Plot OLS coefficients
coef = ols_model.params
errors = ols_model.bse
fig, ax = plt.subplots(figsize=(10, 6))
coef[1:].plot(kind='bar', yerr=errors[1:], ax=ax, color='lightblue', capsize=5)
ax.set_title('OLS Regression Coefficients (Log Export Value)')
ax.set_ylabel('Coefficient (Elasticity)')
ax.set_xlabel('Variables')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('ols_coefficients.png')
plt.close()

# VIF Analysis
X = df[['Auto_Tariff_Percent', 'Z_GDP_Growth', 'Z_Exchange_Rate']].copy()
X['Intercept'] = 1
vif_data = pd.DataFrame()
vif_data['Variable'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\nVariance Inflation Factors:\n", vif_data)


OLS Regression Results (Log Export Value):

                            OLS Regression Results                            
Dep. Variable:       Log_Export_Value   R-squared:                       0.316
Model:                            OLS   Adj. R-squared:                  0.219
Method:                 Least Squares   F-statistic:                     28.91
Date:                Wed, 30 Apr 2025   Prob (F-statistic):            0.00359
Time:                        09:52:06   Log-Likelihood:                -32.304
No. Observations:                  25   AIC:                             72.61
Df Residuals:                      21   BIC:                             77.48
Df Model:                           3                                         
Covariance Type:              cluster                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------

In [7]:
!pip install ipywidgets
import ipywidgets as widgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.1 MB/s eta 0:00:00


In [8]:

# Standardize predictors
gdp_mean, gdp_std = df['India_GDP_Growth_Percent'].mean(), df['India_GDP_Growth_Percent'].std()
exrate_mean, exrate_std = df['INR_USD_Exchange_Rate'].mean(), df['INR_USD_Exchange_Rate'].std()

# 2. Fit OLS Model
ols_formula = 'Log_Export_Value ~ Auto_Tariff_Percent + Textile_Tariff_Percent + Z_GDP_Growth + Z_Exchange_Rate'
ols_model = smf.ols(ols_formula, data=df).fit(cov_type='cluster', cov_kwds={'groups': df['Trade_Partner']})
print("\nOLS Regression Results (Log Export Value):\n")
print(ols_model.summary())

# 3. Predictive Function
def predict_exports(auto_tariff, textile_tariff, gdp_growth, exchange_rate):
    z_gdp = (gdp_growth - gdp_mean) / gdp_std
    z_exrate = (exchange_rate - exrate_mean) / exrate_std
    input_data = pd.DataFrame({
        'Intercept': [1],
        'Auto_Tariff_Percent': [auto_tariff],
        'Textile_Tariff_Percent': [textile_tariff],
        'Z_GDP_Growth': [z_gdp],
        'Z_Exchange_Rate': [z_exrate]
    })
    log_pred = ols_model.predict(input_data)[0]
    pred_export = np.exp(log_pred)  # Convert log to USD billion
    print(f"Predicted Export Value: ${pred_export:.2f} billion")
    return pred_export

# 4. Slider Interface
auto_tariff_slider = widgets.FloatSlider(value=2.5, min=0, max=25, step=0.1, description='Auto Tariff (%):')
textile_tariff_slider = widgets.FloatSlider(value=5.0, min=0, max=25, step=0.1, description='Textile Tariff (%):')
gdp_slider = widgets.FloatSlider(value=6.5, min=4, max=9, step=0.1, description='GDP Growth (%):')
exrate_slider = widgets.FloatSlider(value=83.5, min=75, max=90, step=0.1, description='INR/USD Rate:')

# Interactive prediction
widgets.interact(predict_exports,
                 auto_tariff=auto_tariff_slider,
                 textile_tariff=textile_tariff_slider,
                 gdp_growth=gdp_slider,
                 exchange_rate=exrate_slider)

# Save model parameters
model_params = ols_model.params.to_dict()
with open('model_parameters.txt', 'w') as f:
    f.write(str(model_params))
print("\nModel parameters saved as 'model_parameters.txt'")


OLS Regression Results (Log Export Value):

                            OLS Regression Results                            
Dep. Variable:       Log_Export_Value   R-squared:                       0.800
Model:                            OLS   Adj. R-squared:                  0.760
Method:                 Least Squares   F-statistic:                 2.887e+05
Date:                Wed, 30 Apr 2025   Prob (F-statistic):           3.60e-11
Time:                        09:52:13   Log-Likelihood:                -16.937
No. Observations:                  25   AIC:                             43.87
Df Residuals:                      20   BIC:                             49.97
Df Model:                           4                                         
Covariance Type:              cluster                                         
                             coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------

interactive(children=(FloatSlider(value=2.5, description='Auto Tariff (%):', max=25.0), FloatSlider(value=5.0,…


Model parameters saved as 'model_parameters.txt'
